# Modelo de Estimación: Stranded Capacity en Data Centers de Alta Densidad

## 1. Supuestos Explícitos y Fuentes Citadas
El modelo segmenta los supuestos financieros y operativos según la tecnología de enfriamiento.

**Matriz de Supuestos Termodinámicos:**
*   **Aire Tradicional:** PUE de 1.58, Stranded Capacity del 12% - 13%, TCO de \$6.5M a \$11M por MW.
*   **Híbrido:** PUE de 1.25, Stranded Capacity del 8% - 10%, TCO de \$11M a \$13M por MW.
*   **Líquido (Direct-to-Chip):** PUE de 1.10, Stranded Capacity < 5%, TCO de \$8M a \$14M por MW.

**Fuentes:**
*   *Microsoft GFS (Sankar & Vaid, 2010):* Sobre-provisión de hasta el 13% en base a trazas reales.
*   *Supermicro (2025):* Ahorro de 16% al eliminar IT-fan power en liquid cooling.
*   *Mercado Colocation:* Límite superior financiero de \$184/kW/mes.

In [1]:
class StrandedCapacityModel:
    def __init__(self):
        # Matriz de Supuestos
        self.cooling_matrix = {
            'Air Traditional': {'pue': 1.58, 'stranded_pct_range': (0.12, 0.13), 'tco_10yr_per_mw': (6500000, 11000000)},
            'Híbrido': {'pue': 1.25, 'stranded_pct_range': (0.08, 0.10), 'tco_10yr_per_mw': (11000000, 13000000)},
            'Liquid Direct-to-Chip': {'pue': 1.10, 'stranded_pct_range': (0.01, 0.05), 'tco_10yr_per_mw': (8000000, 14000000)}
        }
        self.colocation_ceiling_usd_mw_year = 184 * 1000 * 12

    def estimate(self, capacity_mw: float, current_utilization: float, cooling_type: str) -> dict:
        params = self.cooling_matrix[cooling_type]
        pue = params['pue']
        factor_min, factor_max = params['stranded_pct_range']
        
        unutilized_capacity = capacity_mw * (1 - current_utilization)
        stranded_mw_min = round(unutilized_capacity * factor_min * pue, 2)
        stranded_mw_max = round(unutilized_capacity * factor_max * pue, 2)
        
        tco_annual_min = params['tco_10yr_per_mw'][0] / 10
        tco_annual_max = params['tco_10yr_per_mw'][1] / 10
        
        loss_min = round(stranded_mw_min * tco_annual_min, 2)
        loss_max = round(stranded_mw_max * tco_annual_max, 2)
        
        colocation_cost = capacity_mw * self.colocation_ceiling_usd_mw_year
        internal_cost_est = (capacity_mw * tco_annual_max)
        margin_vs_colo = round(((colocation_cost - internal_cost_est) / colocation_cost) * 100, 2)
        
        return {
            'stranded_mw_range': (stranded_mw_min, stranded_mw_max),
            'loss_usd_range': (loss_min, loss_max),
            'margin_vs_colocation_pct': margin_vs_colo
        }

## 2. Ejecución y Pruebas del Modelo
A continuación evaluamos un caso base: Un facility de 15 MW operando al 87% de utilización, comparando el impacto financiero entre tecnologías.

In [ ]:
model = StrandedCapacityModel()
capacity = 15.0  # MW
utilization = 0.87  # 87%

print("Resultados para Aire Tradicional:")
print(model.estimate(capacity, utilization, 'Air Traditional'))

print("\nResultados para Líquido (Direct-to-Chip):")
print(model.estimate(capacity, utilization, 'Liquid Direct-to-Chip'))